In [ ]:
import cfe
import scanpy as sc

cfe.settings.backend = "python_function"
cfe.logger.setLevel("DEBUG")
import pandas as pd
import numpy as np
from cfe.data import FateAnnData
import matplotlib.pyplot as plt
import seaborn as sns
#这里尝试仅调用一个接口计算聚合指标
from cfe.metric.calculate_metrics import calculate_metrics

数据准备

In [ ]:
fadata = cfe.data.read_erythroid_lineage(n_obs=500)#用于快速测试
fadata

绘图

In [ ]:
cluster_key = "celltype"
fadata.group_onto_nearest_milestones(cluster_key=cluster_key)  # new cluster color

basis = "umap"
cluster_key_list = ["milestone", cluster_key]
cfe.plot.plot_trajectory(fadata, basis=basis, color=cluster_key_list, curve=False)
cfe.plot.plot_trajectory(fadata, basis=basis, color=cluster_key_list, curve=True)
fadata

In [ ]:
cfe.plot.plot_wrapper(fadata)

添加先验，封装轨迹结构并绘图

In [ ]:
prior_information = {
    "start_id": fadata.obs.index[0],
    "groups_id": fadata.obs[cluster_key].tolist()
}
parameters = {"filter_features": False, "connectivity_cutoff": 0.3}
fadata.add_prior_information(**prior_information)  # add prior information to fadata


method_name_list = ["comp1", "state_comp","paga","cluster_mst","projection_mst"]

for method_name in method_name_list:
    method = cfe.method.FateMethod(method_name=method_name)

    method.infer_trajectory(fadata)

    cfe.plot.plot_trajectory(fadata, basis=basis, color=cluster_key_list)

In [ ]:
parsed_model_name_list = fadata.get_all_model_name() # 解析后的模型名称
model_name_list = fadata.get_all_model_name(parse=False)
parsed_model_name_list, model_name_list

In [ ]:
#这里需要构造新的fadata与计算接口兼容，原生fadata嵌套结构与接口不兼容
new_fadata=cfe.data.FateAnnData(X=fadata.X, obs=fadata.obs, uns=fadata.uns)
#除此之外还要为新的fadata添加必要的属性，与旧的fadata保持一致
for i in model_name_list:
    new_fadata.model_name = i
    new_fadata.milestone_wrapper = fadata.trajectory_history_dict[i]["milestone_wrapper"]
    new_fadata.add_waypoints()
    # 使用单引号避免冲突
    print(f"{i}'s milestone_network is:\n {new_fadata.milestone_wrapper['milestone_network']}")
    print(f"{i}'s progressions is:\n {new_fadata.milestone_wrapper['progressions']}")


prior_information = {"start_id": fadata.obs.index[0], "groups_id": fadata.obs[cluster_key].tolist()}
parameters = {"filter_features": False, "connectivity_cutoff": 0.3}
new_fadata.add_prior_information(**prior_information)  # add prior information to fadata

测试新接口

In [ ]:
new_fadata

In [ ]:
new_fadata.uns

In [ ]:
hist=fadata.uns.get("cfe", {}).get("trajectory_history_dict", {})
hist

In [ ]:
list(hist.keys())

In [ ]:
calculate_metrics(new_fadata, now_model=model_name_list, ref_model="ref")